# 🚗 Zmorda.tn Scraper — Selenium (SPA / AngularJS)
Clicks the `fa-arrow-right` next-page button to navigate, scrapes every page.

In [1]:
# Install dependencies (run once)
!pip install selenium webdriver-manager beautifulsoup4 pandas -q

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
import time
from datetime import date

# ── Config ──────────────────────────────────────────────
URL          = "https://zmorda.tn/voiture"
DELAY        = 2.5   # seconds to wait after each page click
HEADLESS     = True  # set False to watch the browser
# ────────────────────────────────────────────────────────

options = webdriver.ChromeOptions()
if HEADLESS:
    options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
)

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)
wait = WebDriverWait(driver, 15)
print("✅ Browser ready")

✅ Browser ready


In [3]:
def wait_for_cards():
    """Wait until at least one car card is visible on the page."""
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".zsl-card")))


def parse_cards(soup):
    """Extract all listing fields from the current page HTML."""
    cars = []
    for card in soup.select(".zsl-card"):

        def txt(selector):
            el = card.select_one(selector)
            return el.get_text(strip=True) if el else ""

        # Image
        img = card.select_one("img")
        image = img["src"] if img and img.has_attr("src") else ""

        # Meta lines
        meta = card.select(".zsl-meta-secondary")
        generation = meta[0].get_text(strip=True) if len(meta) > 0 else ""
        engine     = meta[1].get_text(strip=True) if len(meta) > 1 else ""
        body       = meta[2].get_text(strip=True) if len(meta) > 2 else ""

        # Specs
        specs = card.select(".zsl-spec")
        mileage = specs[0].get_text(strip=True) if len(specs) > 0 else ""
        fuel    = specs[1].get_text(strip=True) if len(specs) > 1 else ""
        gearbox = specs[2].get_text(strip=True) if len(specs) > 2 else ""

        # Listing URL
        link = card.select_one("a[href]")
        url  = "https://zmorda.tn" + link["href"] if link else ""

        cars.append({
            "title":      txt(".zsl-card__title"),
            "price":      txt(".zsl-card__price"),
            "badge":      txt(".zsl-badge"),
            "generation": generation,
            "engine":     engine,
            "body":       body,
            "mileage":    mileage,
            "fuel":       fuel,
            "gearbox":    gearbox,
            "image":      image,
            "url":        url,
        })
    return cars


def get_next_button():
    """Return the next-page button element, or None if disabled/absent."""
    try:
        btn = driver.find_element(
            By.CSS_SELECTOR,
            "button[ng-click='nextPage()']"  # exact match from the HTML you shared
        )
        disabled = btn.get_attribute("disabled")
        return None if disabled else btn
    except Exception:
        return None


print("✅ Helpers defined")

✅ Helpers defined


In [4]:
all_cars = []
page_num = 1

# Open the listing page
driver.get(URL)
wait_for_cards()
time.sleep(DELAY)  # let Angular fully render

while True:
    print(f"Scraping page {page_num} ...", end=" ")
    soup  = BeautifulSoup(driver.page_source, "html.parser")
    cards = parse_cards(soup)
    all_cars.extend(cards)
    print(f"{len(cards)} listings  (total so far: {len(all_cars)})")

    # Try to click the next-page arrow
    next_btn = get_next_button()
    if next_btn is None:
        print("\n🏁 Last page reached — done!")
        break

    # Scroll button into view and click
    driver.execute_script("arguments[0].scrollIntoView(true);", next_btn)
    time.sleep(0.3)
    next_btn.click()

    # Wait for Angular to re-render the new page of cards
    time.sleep(DELAY)
    wait_for_cards()
    page_num += 1

driver.quit()
print(f"\n✅ Browser closed. Total listings: {len(all_cars)}")

Scraping page 1 ... 25 listings  (total so far: 25)
Scraping page 2 ... 25 listings  (total so far: 50)
Scraping page 3 ... 25 listings  (total so far: 75)
Scraping page 4 ... 25 listings  (total so far: 100)
Scraping page 5 ... 25 listings  (total so far: 125)
Scraping page 6 ... 25 listings  (total so far: 150)
Scraping page 7 ... 25 listings  (total so far: 175)
Scraping page 8 ... 25 listings  (total so far: 200)
Scraping page 9 ... 25 listings  (total so far: 225)
Scraping page 10 ... 25 listings  (total so far: 250)
Scraping page 11 ... 25 listings  (total so far: 275)
Scraping page 12 ... 25 listings  (total so far: 300)
Scraping page 13 ... 25 listings  (total so far: 325)
Scraping page 14 ... 25 listings  (total so far: 350)
Scraping page 15 ... 25 listings  (total so far: 375)
Scraping page 16 ... 25 listings  (total so far: 400)
Scraping page 17 ... 25 listings  (total so far: 425)
Scraping page 18 ... 25 listings  (total so far: 450)
Scraping page 19 ... 25 listings  (total

In [5]:
df = pd.DataFrame(all_cars)

# Clean price → integer DT
df["price_dt"] = (
    df["price"].str.replace(r"[^\d]", "", regex=True)
    .replace("", None).astype("Int64")
)

# Clean mileage → integer km
df["mileage_km"] = (
    df["mileage"].str.replace(r"[^\d]", "", regex=True)
    .replace("", None).astype("Int64")
)

print(df.shape)
df.head(10)

(1152, 13)


,title,price,badge,generation,engine,body,mileage,fuel,gearbox,image,url,price_dt,mileage_km
0,Ford Ranger,100 000 DT,Première main,2.2 TDCi BVA AWD (160 cv),pickup Double Cab pickup 4-portes,,"240,000km",Diesel,Automatique,https://zmorda-bucket.s3.eu-west-3.amazonaws.c...,,100000,240000
1,Peugeot 106,9 500 DT,,1.1 BVM (60 cv),citadine Hatchback 5-portes,,"265,000km",Essence,Manuelle,https://zmorda-bucket.s3.eu-west-3.amazonaws.c...,,9500,265000
2,Citroen C3,33 000 DT,,1.2 BVM (82 cv),citadine Hatchback 5-portes,,"109,000km",Essence,Manuelle,https://zmorda-bucket.s3.eu-west-3.amazonaws.c...,,33000,109000
3,Hyundai i10,42 500 DT,,1.2 BVA (84 cv),citadine Hayon 5 portes,,"119,000km",Essence,Automatique,https://zmorda-bucket.s3.eu-west-3.amazonaws.c...,,42500,119000
4,Volkswagen Polo,43 900 DT,,1.0 MPI BVM (75 cv),citadine Hatchback,,"185,000km",Essence,Manuelle,https://zmorda-bucket.s3.eu-west-3.amazonaws.c...,,43900,185000
5,Chevrolet Cruze,26 000 DT,,1.6 BVM (109 cv),berline Berline 4-portes,,"250,000km",Essence,Manuelle,https://zmorda-bucket.s3.eu-west-3.amazonaws.c...,,26000,250000
6,Tata Xenon,35 000 DT,Première main,2.2 BVM 4WD (140 cv),pickup Pickup Double cabine,,"360,000km",Diesel,Manuelle,https://zmorda-bucket.s3.eu-west-3.amazonaws.c...,,35000,360000
7,Fiat 500,26 000 DT,,POP 1.2 BVA (69 cv),citadine Hatchback,,"115,000km",Essence,Automatique,https://zmorda-bucket.s3.eu-west-3.amazonaws.c...,,26000,115000
8,Volkswagen Polo,40 000 DT,Première main,1.4 BVA (125 cv),citadine liftback,,160km,Essence,Manuelle,https://zmorda-bucket.s3.eu-west-3.amazonaws.c...,,40000,160
9,Renault Clio,14 000 DT,,1.4 BVM (98 cv),citadine Hayon 5 portes,,"300,000km",Essence,Manuelle,https://zmorda-bucket.s3.eu-west-3.amazonaws.c...,,14000,300000


In [6]:
filename = f"zmorda_{date.today()}.csv"
df.to_csv(filename, index=False, encoding="utf-8-sig")
print(f"💾 Saved → {filename}")

💾 Saved → zmorda_2026-08-20.csv


In [7]:
print("=== Quick Stats ===")
print(f"Total listings : {len(df)}")
print(f"Première main  : {(df['badge'] == 'PREMIÈRE MAIN').sum()}")
print(f"Price range    : {df['price_dt'].min():,} – {df['price_dt'].max():,} DT")
print(f"Avg price      : {df['price_dt'].mean():,.0f} DT")
print()
print("Top 10 brands:")
print(df["title"].str.split().str[0].value_counts().head(10))
print()
print("Fuel breakdown:")
print(df["fuel"].value_counts())

=== Quick Stats ===
Total listings : 1152
Première main  : 0
Price range    : 1 – 48,000,000 DT
Avg price      : 89,752 DT

Top 10 brands:
title
Volkswagen       160
Peugeot          122
Renault           95
Citroen           90
Kia               76
Fiat              72
Hyundai           58
Toyota            40
SEAT              39
Mercedes-Benz     38
Name: count, dtype: int64

Fuel breakdown:
fuel
Essence       854
Diesel        254
Hybride        34
Electrique      5
                3
Électrique      2
Name: count, dtype: int64
